In [ ]:
# ============================================================
# VishGuard — External Dataset Acquisition
# Kaggle Voice Phishing Dataset
# ============================================================

import os
import zipfile
import requests
from pathlib import Path

KAGGLE_URL = "https://www.kaggle.com/api/v1/datasets/download/maxwellkuria/scamdataset"

DOWNLOAD_DIR = Path("/content/vishguard_external")
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

zip_path = DOWNLOAD_DIR / "scamdataset.zip"

print("Downloading Kaggle dataset...")

response = requests.get(KAGGLE_URL, stream=True)

print("HTTP status:", response.status_code)

if response.status_code != 200:
    raise RuntimeError(
        f"Download failed with HTTP status {response.status_code}"
    )

with open(zip_path, "wb") as f:
    for chunk in response.iter_content(chunk_size=1024 * 1024):
        if chunk:
            f.write(chunk)

print(f"\nDownloaded successfully.")
print(f"ZIP file: {zip_path}")
print(f"Size: {zip_path.stat().st_size / (1024**2):.2f} MB")

HTTP status: 200

Downloaded successfully.
ZIP file: /content/vishguard_external/scamdataset.zip
Size: 0.73 MB


In [ ]:
# ============================================================
# VishGuard — Inspect External Kaggle Dataset
# ============================================================

import zipfile
from pathlib import Path

extract_dir = Path("/content/vishguard_external/scamdataset_extracted")
extract_dir.mkdir(parents=True, exist_ok=True)

# Extract
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_dir)

print("Extracted files:\n")

for path in sorted(extract_dir.rglob("*")):
    if path.is_file():
        print(f"- {path.relative_to(extract_dir)} | "
              f"{path.stat().st_size / 1024:.2f} KB")

Extracted files:

- Scam Dataset calls/Normal_Converstion.txt | 2071.25 KB
- Scam Dataset calls/Scam_converstions.txt | 2501.42 KB


In [ ]:
# ============================================================
# VishGuard — External Dataset Content Inspection
# ============================================================

normal_path = extract_dir / "Scam Dataset calls" / "Normal_Converstion.txt"
scam_path = extract_dir / "Scam Dataset calls" / "Scam_converstions.txt"

def inspect_txt(path, name, n_chars=3000):
    print("=" * 80)
    print(name)
    print("=" * 80)

    with open(path, "r", encoding="utf-8", errors="replace") as f:
        text = f.read()

    print(f"File size: {len(text):,} characters")
    print(f"Number of lines: {len(text.splitlines()):,}")
    print("\nFirst content:\n")
    print(text[:n_chars])
    print("\n" + "=" * 80)


inspect_txt(normal_path, "NORMAL CONVERSATIONS")
inspect_txt(scam_path, "SCAM CONVERSATIONS")

NORMAL CONVERSATIONS
File size: 2,101,611 characters
Number of lines: 13,477

First content:

Mshukiwa: Habari, huyu ni Karen kutoka ofisi ya Dk. Smith. Ninakupigia ili kuthibitisha miadi yako ya kesho saa 2 Usiku. Innocent: Oh, habari Karen. Nadhani kunaweza kuwa na makosa. Sina miadi na Dk. Smith kesho. Mtuhumiwa: Sawa, ngoja niangalie hilo. Tafadhali unaweza kunithibitishia jina lako na tarehe ya kuzaliwa? Innocent: Subiri, umepataje namba yangu? Sikumbuki kuweka miadi au kutoa maelezo yangu ya mawasiliano. Mtuhumiwa: Naomba radhi kwa kuchanganyikiwa. Lazima tumepata nambari yako kutoka kwa mtoa huduma wako wa bima. Innocent: Sinunui. Wewe si mzuri sana katika hili, sivyo? Mtuhumiwa: Um, sijui unamaanisha nini. Ninajaribu tu kufanya kazi yangu. Innocent: Ihifadhi. Wewe si tapeli, sivyo? Wewe ni kweli kutoka ofisi ya Dk. Smith. Mtuhumiwa: Ah, ndio. mimi niko. Naomba radhi kwa kuchanganyikiwa hapo awali. Tulikuwa na mawasiliano mabaya. Innocent: Ha! Nilijua. Wewe si mjanja vya kutosha

In [ ]:
# ============================================================
# VishGuard — Conversation Structure & Duplicate Analysis
# ============================================================

import re
from collections import Counter

def load_text(path):
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        return f.read()

normal_text = load_text(normal_path)
scam_text = load_text(scam_path)

# ------------------------------------------------------------
# 1. Detect conversation boundaries
# ------------------------------------------------------------
# Each conversation appears to begin with a speaker label.
# We inspect several possible boundary patterns.

speaker_pattern = r"(?:^|\n)(?:Mshukiwa|Mtuhumiwa|Mshukiwa):"

normal_starts = re.findall(speaker_pattern, normal_text)
scam_starts = re.findall(speaker_pattern, scam_text)

print("=" * 80)
print("CONVERSATION STRUCTURE")
print("=" * 80)

print("Normal speaker starts detected:", len(normal_starts))
print("Scam speaker starts detected:", len(scam_starts))

# ------------------------------------------------------------
# 2. Count lines containing speaker labels
# ------------------------------------------------------------

normal_lines = normal_text.splitlines()
scam_lines = scam_text.splitlines()

normal_speaker_lines = [
    line for line in normal_lines
    if re.search(r"(Mshukiwa|Mtuhumiwa|Innocent):", line)
]

scam_speaker_lines = [
    line for line in scam_lines
    if re.search(r"(Mshukiwa|Mtuhumiwa|Innocent):", line)
]

print("\nNormal lines with speaker labels:", len(normal_speaker_lines))
print("Scam lines with speaker labels:", len(scam_speaker_lines))

# ------------------------------------------------------------
# 3. Estimate conversations using blank-line separation
# ------------------------------------------------------------

normal_blocks = [
    block.strip()
    for block in re.split(r"\n\s*\n", normal_text)
    if block.strip()
]

scam_blocks = [
    block.strip()
    for block in re.split(r"\n\s*\n", scam_text)
    if block.strip()
]

print("\nNormal blocks:", len(normal_blocks))
print("Scam blocks:", len(scam_blocks))

# ------------------------------------------------------------
# 4. Basic length statistics
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("LENGTH STATISTICS")
print("=" * 80)

for name, blocks in [
    ("Normal", normal_blocks),
    ("Scam", scam_blocks)
]:
    lengths = [len(x) for x in blocks]
    words = [len(x.split()) for x in blocks]

    print(f"\n{name}:")
    print(f"  Samples: {len(blocks)}")
    print(f"  Mean characters: {sum(lengths)/len(lengths):.1f}")
    print(f"  Median characters: {sorted(lengths)[len(lengths)//2]}")
    print(f"  Mean words: {sum(words)/len(words):.1f}")
    print(f"  Median words: {sorted(words)[len(words)//2]}")

# ------------------------------------------------------------
# 5. Exact duplicate analysis
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("EXACT DUPLICATES")
print("=" * 80)

for name, blocks in [
    ("Normal", normal_blocks),
    ("Scam", scam_blocks)
]:
    counts = Counter(blocks)
    duplicates = sum(1 for c in counts.values() if c > 1)
    repeated_instances = sum(c - 1 for c in counts.values() if c > 1)

    print(f"\n{name}:")
    print("  Unique samples:", len(counts))
    print("  Repeated sample patterns:", duplicates)
    print("  Duplicate instances:", repeated_instances)

# ------------------------------------------------------------
# 6. Show first 3 detected blocks
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FIRST 3 NORMAL BLOCKS")
print("=" * 80)

for i, block in enumerate(normal_blocks[:3], 1):
    print(f"\n--- NORMAL {i} ---")
    print(block[:1500])

print("\n" + "=" * 80)
print("FIRST 3 SCAM BLOCKS")
print("=" * 80)

for i, block in enumerate(scam_blocks[:3], 1):
    print(f"\n--- SCAM {i} ---")
    print(block[:1500])

CONVERSATION STRUCTURE
Normal speaker starts detected: 640
Scam speaker starts detected: 640

Normal lines with speaker labels: 1280
Scam lines with speaker labels: 1280

Normal blocks: 2998
Scam blocks: 401

LENGTH STATISTICS

Normal:
  Samples: 2998
  Mean characters: 699.0
  Median characters: 199
  Mean words: 114.1
  Median words: 34

Scam:
  Samples: 401
  Mean characters: 6370.2
  Median characters: 314
  Mean words: 1011.1
  Median words: 47

EXACT DUPLICATES

Normal:
  Unique samples: 2725
  Repeated sample patterns: 260
  Duplicate instances: 273

Scam:
  Unique samples: 401
  Repeated sample patterns: 0
  Duplicate instances: 0

FIRST 3 NORMAL BLOCKS

--- NORMAL 1 ---
Mshukiwa: Habari, huyu ni Karen kutoka ofisi ya Dk. Smith. Ninakupigia ili kuthibitisha miadi yako ya kesho saa 2 Usiku. Innocent: Oh, habari Karen. Nadhani kunaweza kuwa na makosa. Sina miadi na Dk. Smith kesho. Mtuhumiwa: Sawa, ngoja niangalie hilo. Tafadhali unaweza kunithibitishia jina lako na tarehe ya kuz

In [ ]:
# ============================================================
# VishGuard — External Dataset Quality Analysis
# ============================================================

import re
import pandas as pd
from collections import Counter

# ------------------------------------------------------------
# 1. Create preliminary dataset
# ------------------------------------------------------------

external_rows = []

for label, blocks in [
    ("legitimate", normal_blocks),
    ("fraudulent", scam_blocks)
]:
    for i, text in enumerate(blocks):
        external_rows.append({
            "external_id": f"{label}_{i+1:05d}",
            "type": label,
            "raw_text": text
        })

external_df = pd.DataFrame(external_rows)

# ------------------------------------------------------------
# 2. Basic text statistics
# ------------------------------------------------------------

external_df["char_length"] = external_df["raw_text"].str.len()
external_df["word_count"] = external_df["raw_text"].str.split().str.len()

# ------------------------------------------------------------
# 3. Simple English / Swahili detection
# ------------------------------------------------------------

swahili_words = {
    "na", "ni", "huyu", "habari", "sawa", "yako", "yangu",
    "kwa", "katika", "una", "unaweza", "tafadhali", "nambari",
    "simu", "pesa", "akaunti", "benki", "tunapiga", "kupiga",
    "maelezo", "mimi", "wewe", "sisi", "hii", "hiyo", "sana",
    "asante", "karibu", "kesho", "leo", "tuna", "kutoa",
    "kufanya", "jina", "mtu", "muda"
}

def detect_language_simple(text):
    words = set(re.findall(r"\b[a-zA-Z]+\b", text.lower()))
    sw_count = len(words.intersection(swahili_words))

    if sw_count >= 3:
        return "sw"

    return "en"

external_df["detected_language"] = external_df["raw_text"].apply(
    detect_language_simple
)

# ------------------------------------------------------------
# 4. Dataset overview
# ------------------------------------------------------------

print("=" * 80)
print("EXTERNAL DATASET OVERVIEW")
print("=" * 80)

print("\nTotal preliminary samples:", len(external_df))

print("\nClass distribution:")
print(external_df["type"].value_counts())

print("\nLanguage distribution:")
print(external_df["detected_language"].value_counts())

print("\nClass × language:")
print(
    pd.crosstab(
        external_df["type"],
        external_df["detected_language"]
    )
)

# ------------------------------------------------------------
# 5. Length statistics
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("LENGTH STATISTICS BY CLASS")
print("=" * 80)

stats = external_df.groupby("type")[["char_length", "word_count"]].agg(
    ["count", "mean", "median", "min", "max"]
)

print(stats)

# ------------------------------------------------------------
# 6. Longest samples
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("TOP 20 LONGEST SAMPLES")
print("=" * 80)

longest = external_df.sort_values(
    "word_count",
    ascending=False
).head(20)

print(
    longest[
        [
            "external_id",
            "type",
            "detected_language",
            "word_count",
            "char_length"
        ]
    ].to_string(index=False)
)

# ------------------------------------------------------------
# 7. Very short samples
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("VERY SHORT SAMPLES (< 10 WORDS)")
print("=" * 80)

short_samples = external_df[
    external_df["word_count"] < 10
]

print("Count:", len(short_samples))

print(
    short_samples[
        [
            "external_id",
            "type",
            "detected_language",
            "word_count",
            "raw_text"
        ]
    ].head(30).to_string(index=False)
)

# ------------------------------------------------------------
# 8. Exact duplicates
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("EXACT DUPLICATES")
print("=" * 80)

for label in ["legitimate", "fraudulent"]:

    subset = external_df[
        external_df["type"] == label
    ]

    duplicate_instances = subset["raw_text"].duplicated().sum()

    unique_count = subset["raw_text"].nunique()

    print(f"\n{label}:")
    print("  Total:", len(subset))
    print("  Unique:", unique_count)
    print("  Duplicate instances:", duplicate_instances)

# ------------------------------------------------------------
# 9. Final preview
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("DATASET READY FOR FURTHER INSPECTION")
print("=" * 80)

print(external_df.head())
print("\nColumns:")
print(external_df.columns.tolist())

EXTERNAL DATASET OVERVIEW

Total preliminary samples: 3399

Class distribution:
type
legitimate    2998
fraudulent     401
Name: count, dtype: int64

Language distribution:
detected_language
sw    1820
en    1579
Name: count, dtype: int64

Class × language:
detected_language    en    sw
type                         
fraudulent          300   101
legitimate         1279  1719

LENGTH STATISTICS BY CLASS
           char_length                                  word_count  \
                 count        mean median  min      max      count   
type                                                                 
fraudulent         401  6370.15212  314.0   98  2464202        401   
legitimate        2998   699.00467  199.0  161  1498362       2998   

                                            
                   mean median min     max  
type                                        
fraudulent  1011.104738   47.0  14  390853  
legitimate   114.064710   34.0  25  241089  

TOP 20 LONGEST SA

In [ ]:
# ============================================================
# VishGuard — Discover Conversation Separators
# ============================================================

import re

def inspect_boundaries(path, name):
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        text = f.read()

    print("=" * 80)
    print(name)
    print("=" * 80)

    # Show newline statistics
    print("Total characters:", len(text))
    print("Total \\n:", text.count("\n"))
    print("Total \\r:", text.count("\r"))
    print("Total blank-line separators:", len(re.findall(r"\n\s*\n", text)))

    # Look for common separator patterns
    patterns = {
        "=== separators": r"={3,}",
        "--- separators": r"-{3,}",
        "Numbered headings": r"(?:^|\n)\s*\d+[\.\):\-]",
        "Conversation headings": r"(?i)(?:^|\n)\s*(?:conversation|dialogue|call)\s*\d*",
        "Speaker after newline": r"\n\s*(?:Caller|Receiver|Mshukiwa|Mtuhumiwa|Innocent)\s*:",
    }

    print("\nPattern counts:")
    for label, pattern in patterns.items():
        print(f"{label}: {len(re.findall(pattern, text))}")

    # --------------------------------------------------------
    # Print selected portions around newline boundaries
    # --------------------------------------------------------

    print("\n" + "=" * 80)
    print("SAMPLE RAW BOUNDARIES")
    print("=" * 80)

    # First 20 newline-delimited lines
    lines = text.splitlines()

    print("\nFirst 20 lines:")
    for i, line in enumerate(lines[:20], 1):
        print(f"[{i}] {line[:500]}")

    # Lines around 600-660
    print("\n" + "=" * 80)
    print("LINES AROUND THE 640-SPEAKER REGION")
    print("=" * 80)

    for i, line in enumerate(lines[620:660], 621):
        print(f"[{i}] {line[:700]}")

    return text


normal_raw = inspect_boundaries(
    normal_path,
    "NORMAL FILE"
)

scam_raw = inspect_boundaries(
    scam_path,
    "SCAM FILE"
)

NORMAL FILE
Total characters: 2101611
Total \n: 13477
Total \r: 0
Total blank-line separators: 2997

Pattern counts:
=== separators: 0
--- separators: 0
Numbered headings: 0
Conversation headings: 6000
Speaker after newline: 9639

SAMPLE RAW BOUNDARIES

First 20 lines:
[1] Mshukiwa: Habari, huyu ni Karen kutoka ofisi ya Dk. Smith. Ninakupigia ili kuthibitisha miadi yako ya kesho saa 2 Usiku. Innocent: Oh, habari Karen. Nadhani kunaweza kuwa na makosa. Sina miadi na Dk. Smith kesho. Mtuhumiwa: Sawa, ngoja niangalie hilo. Tafadhali unaweza kunithibitishia jina lako na tarehe ya kuzaliwa? Innocent: Subiri, umepataje namba yangu? Sikumbuki kuweka miadi au kutoa maelezo yangu ya mawasiliano. Mtuhumiwa: Naomba radhi kwa kuchanganyikiwa. Lazima tumepata nambari yako kuto
[2] Mtuhumiwa: Hi, huyu ni John? Innocent: Ndio mimi. Huyu ni nani? Mshukiwa: Huyu ni Karen kutoka ofisi ya Dk. Smith. Ninakupigia ili kuthibitisha miadi yako ya kesho saa 2 Usiku. Innocent: Ah, ndio. Nadhani nimepanga hiyo. 

In [ ]:
# ============================================================
# VishGuard — Analyze Language Sections and Conversation Structure
# ============================================================

import re

def analyze_structure(text, name):
    lines = text.splitlines()

    print("=" * 80)
    print(name)
    print("=" * 80)

    # --------------------------------------------------------
    # 1. Detect language transitions using speaker labels
    # --------------------------------------------------------

    swahili_speakers = [
        "Mshukiwa:",
        "Mtuhumiwa:",
        "Innocent:"
    ]

    english_speakers = [
        "Suspect:",
        "Innocent:"
    ]

    sw_indices = []
    en_indices = []

    for i, line in enumerate(lines):
        if any(line.strip().startswith(s) for s in swahili_speakers):
            sw_indices.append(i + 1)

        if line.strip().startswith("Suspect:"):
            en_indices.append(i + 1)

    print("Total lines:", len(lines))

    if sw_indices:
        print("First Swahili speaker line:", min(sw_indices))

    if en_indices:
        print("First English speaker line:", min(en_indices))

    # --------------------------------------------------------
    # 2. Show language transition region
    # --------------------------------------------------------

    transition_points = []

    for i, line in enumerate(lines):
        if line.strip().startswith("Suspect:"):
            transition_points.append(i + 1)

    if transition_points:
        first_en = transition_points[0]

        print("\nFirst English conversation begins around line:", first_en)

        print("\nLines around language transition:")
        start = max(0, first_en - 5)
        end = min(len(lines), first_en + 5)

        for j in range(start, end):
            print(f"[{j+1}] {lines[j][:1000]}")

    # --------------------------------------------------------
    # 3. Blank-line structure
    # --------------------------------------------------------

    blocks = re.split(r"\n\s*\n", text)

    blocks = [b.strip() for b in blocks if b.strip()]

    print("\nNon-empty blank-line blocks:", len(blocks))

    lengths = [len(b) for b in blocks]

    if lengths:
        print("Block length:")
        print("  Mean:", round(sum(lengths) / len(lengths), 2))
        print("  Median:", sorted(lengths)[len(lengths)//2])
        print("  Min:", min(lengths))
        print("  Max:", max(lengths))

    # --------------------------------------------------------
    # 4. Find suspiciously large blocks
    # --------------------------------------------------------

    large_blocks = [
        (i + 1, len(b), len(b.split()))
        for i, b in enumerate(blocks)
        if len(b) > 10000
    ]

    print("\nBlocks > 10,000 characters:", len(large_blocks))

    for item in large_blocks[:10]:
        print("  Block", item[0],
              "| chars:", item[1],
              "| words:", item[2])

    # --------------------------------------------------------
    # 5. Speaker counts
    # --------------------------------------------------------

    speaker_patterns = {
        "Mshukiwa": r"\bMshukiwa:",
        "Mtuhumiwa": r"\bMtuhumiwa:",
        "Suspect": r"\bSuspect:",
        "Innocent": r"\bInnocent:"
    }

    print("\nSpeaker counts:")

    for speaker, pattern in speaker_patterns.items():
        print(f"  {speaker}: {len(re.findall(pattern, text))}")

    # --------------------------------------------------------
    # 6. Inspect block sizes around beginning
    # --------------------------------------------------------

    print("\nFirst 10 block sizes:")

    for i, block in enumerate(blocks[:10], 1):
        print(
            f"  Block {i}: "
            f"{len(block)} chars | "
            f"{len(block.split())} words"
        )

    # --------------------------------------------------------
    # 7. Inspect block sizes around transition
    # --------------------------------------------------------

    print("\nBlocks around the first English section:")

    for i, block in enumerate(blocks):
        if "Suspect:" in block:
            start = max(0, i - 3)
            end = min(len(blocks), i + 3)

            for j in range(start, end):
                preview = blocks[j][:250].replace("\n", " ")
                print(
                    f"  Block {j+1}: "
                    f"{len(blocks[j])} chars | "
                    f"{preview}..."
                )

            break

    return blocks


normal_blocks = analyze_structure(
    normal_raw,
    "NORMAL FILE"
)

scam_blocks = analyze_structure(
    scam_raw,
    "SCAM FILE"
)

NORMAL FILE
Total lines: 13477
First Swahili speaker line: 1
First English speaker line: 641

First English conversation begins around line: 641

Lines around language transition:
[637] Mshukiwa: Hujambo, hii ni huduma kwa wateja kwa Benki ya Amerika? Innocent: Ah, ndio, umefikia nambari inayofaa. Nikusaidie nini leo? Mshukiwa: Nilipigiwa simu na mtu anayedai kuwa anatoka katika idara yako ya utapeli na akasema akaunti yangu imeingiliwa. Innocent: Sawa, ngoja nikuchunguze. Unaweza kunipa nambari yako ya akaunti ili niiangalie? Mshukiwa: Subiri, nitajuaje kuwa wewe ni kutoka Benki ya Amerika? Unaweza kuwa mtu yeyote. Innocent: Swali zuri sana! Kwa kweli, mimi si kutoka Benki ya Amerika. Umefikia nambari isiyo sahihi. Mimi ni mtu wa kawaida tu. Mtuhumiwa: Lo, samahani sana! Lazima nimepiga nambari isiyo sahihi. Innocent: Hakuna wasiwasi hata kidogo! Inatokea kwa bora wetu. Lakini lazima niseme, nimefurahishwa kuwa ulikuwa mwangalifu na hukutoa habari zozote za kibinafsi. Ulifanya kila ki

In [ ]:
# ============================================================
# VishGuard — Inspect Raw Dataset Boundaries
# ============================================================

print("=" * 80)
print("NORMAL — FIRST 15 RAW LINES")
print("=" * 80)

for i, line in enumerate(normal_raw.splitlines()[:15], 1):
    print(f"\n[{i}] {line[:1200]}")


print("\n\n" + "=" * 80)
print("NORMAL — LINES 635 TO 650")
print("=" * 80)

normal_lines = normal_raw.splitlines()

for i in range(634, 650):
    print(f"\n[{i+1}] {normal_lines[i][:1200]}")


print("\n\n" + "=" * 80)
print("SCAM — FIRST 15 RAW LINES")
print("=" * 80)

for i, line in enumerate(scam_raw.splitlines()[:15], 1):
    print(f"\n[{i}] {line[:1200]}")


print("\n\n" + "=" * 80)
print("SCAM — LINES 635 TO 650")
print("=" * 80)

scam_lines = scam_raw.splitlines()

for i in range(634, 650):
    print(f"\n[{i+1}] {scam_lines[i][:1200]}")

NORMAL — FIRST 15 RAW LINES

[1] Mshukiwa: Habari, huyu ni Karen kutoka ofisi ya Dk. Smith. Ninakupigia ili kuthibitisha miadi yako ya kesho saa 2 Usiku. Innocent: Oh, habari Karen. Nadhani kunaweza kuwa na makosa. Sina miadi na Dk. Smith kesho. Mtuhumiwa: Sawa, ngoja niangalie hilo. Tafadhali unaweza kunithibitishia jina lako na tarehe ya kuzaliwa? Innocent: Subiri, umepataje namba yangu? Sikumbuki kuweka miadi au kutoa maelezo yangu ya mawasiliano. Mtuhumiwa: Naomba radhi kwa kuchanganyikiwa. Lazima tumepata nambari yako kutoka kwa mtoa huduma wako wa bima. Innocent: Sinunui. Wewe si mzuri sana katika hili, sivyo? Mtuhumiwa: Um, sijui unamaanisha nini. Ninajaribu tu kufanya kazi yangu. Innocent: Ihifadhi. Wewe si tapeli, sivyo? Wewe ni kweli kutoka ofisi ya Dk. Smith. Mtuhumiwa: Ah, ndio. mimi niko. Naomba radhi kwa kuchanganyikiwa hapo awali. Tulikuwa na mawasiliano mabaya. Innocent: Ha! Nilijua. Wewe si mjanja vya kutosha kuwa tapeli. Asante kwa kuthibitisha.

[2] Mtuhumiwa: Hi, hu

In [ ]:
# ============================================================
# VishGuard — Map Dataset Sections
# ============================================================

normal_lines = [x.strip() for x in normal_raw.splitlines() if x.strip()]
scam_lines = [x.strip() for x in scam_raw.splitlines() if x.strip()]

print("=" * 80)
print("NORMAL DATASET STRUCTURE")
print("=" * 80)

print("Total non-empty lines:", len(normal_lines))

# Check selected positions
for pos in [1, 2, 639, 640, 641, 642, 1279, 1280, 1281, 1282, 2000, 3000]:
    if pos <= len(normal_lines):
        line = normal_lines[pos - 1]
        print(f"\n[{pos}] {line[:180]}")


print("\n" + "=" * 80)
print("SCAM DATASET STRUCTURE")
print("=" * 80)

print("Total non-empty lines:", len(scam_lines))

for pos in [1, 2, 639, 640, 641, 642, 1279, 1280, 1281, 1282, 2000, 3000, 4000, 5000]:
    if pos <= len(scam_lines):
        line = scam_lines[pos - 1]
        print(f"\n[{pos}] {line[:180]}")


print("\n" + "=" * 80)
print("LANGUAGE / SPEAKER AT SECTION BOUNDARIES")
print("=" * 80)

def first_speaker(line):
    m = re.match(r"^(Mshukiwa|Mtuhumiwa|Suspect|Innocent):", line)
    return m.group(1) if m else "UNKNOWN"


print("\nNORMAL:")
for pos in [1, 640, 641, 1280, 1281, 2000, 3000]:
    if pos <= len(normal_lines):
        print(pos, "->", first_speaker(normal_lines[pos-1]))

print("\nSCAM:")
for pos in [1, 640, 641, 1280, 1281, 2000, 3000, 4000, 5000]:
    if pos <= len(scam_lines):
        print(pos, "->", first_speaker(scam_lines[pos-1]))

NORMAL DATASET STRUCTURE
Total non-empty lines: 10480

[1] Mshukiwa: Habari, huyu ni Karen kutoka ofisi ya Dk. Smith. Ninakupigia ili kuthibitisha miadi yako ya kesho saa 2 Usiku. Innocent: Oh, habari Karen. Nadhani kunaweza kuwa na makosa

[2] Mtuhumiwa: Hi, huyu ni John? Innocent: Ndio mimi. Huyu ni nani? Mshukiwa: Huyu ni Karen kutoka ofisi ya Dk. Smith. Ninakupigia ili kuthibitisha miadi yako ya kesho saa 2 Usiku. Inn

[639] Mshukiwa: Habari, namtafuta John Smith. Innocent: Mimi sio John Smith. Lakini ninaweza kujaribu kukusaidia kwa chochote unachohitaji. Hii inahusu nini? Mshukiwa: Lo, nilitakiwa kuk

[640] Mshukiwa: Hujambo, je, huu ndio usaidizi wa kiufundi kwa Microsoft? Innocent: Ndio hawa. Je, inaonekana kuwa tatizo kwenye kompyuta yako ni nini? Mshukiwa: Nimekuwa nikipata madiri

[641] Suspect: Hi, this is Karen from Dr. Smith's office. I'm calling to confirm your appointment for tomorrow at 2 PM.   Innocent: Oh, hi Karen. I think there might be some mistake. I d

[642] Sus

In [ ]:
# ============================================================
# VishGuard — Verify Swahili ↔ English Conversation Alignment
# ============================================================

from difflib import SequenceMatcher

normal_sw = normal_lines[:640]
normal_en = normal_lines[640:1280]

scam_sw = scam_lines[:640]
scam_en = scam_lines[640:1280]

print("=" * 80)
print("NORMAL ALIGNMENT CHECK")
print("=" * 80)

for i in [0, 1, 9, 99, 319, 639]:
    print(f"\nPAIR {i+1}")
    print("SW:", normal_sw[i][:300])
    print("EN:", normal_en[i][:300])

print("\n" + "=" * 80)
print("SCAM ALIGNMENT CHECK")
print("=" * 80)

for i in [0, 1, 9, 99, 319, 639]:
    print(f"\nPAIR {i+1}")
    print("SW:", scam_sw[i][:300])
    print("EN:", scam_en[i][:300])

print("\n" + "=" * 80)
print("BASIC STRUCTURE")
print("=" * 80)

print("Normal SW:", len(normal_sw))
print("Normal EN:", len(normal_en))
print("Scam SW:", len(scam_sw))
print("Scam EN:", len(scam_en))

NORMAL ALIGNMENT CHECK

PAIR 1
SW: Mshukiwa: Habari, huyu ni Karen kutoka ofisi ya Dk. Smith. Ninakupigia ili kuthibitisha miadi yako ya kesho saa 2 Usiku. Innocent: Oh, habari Karen. Nadhani kunaweza kuwa na makosa. Sina miadi na Dk. Smith kesho. Mtuhumiwa: Sawa, ngoja niangalie hilo. Tafadhali unaweza kunithibitishia jina lako na t
EN: Suspect: Hi, this is Karen from Dr. Smith's office. I'm calling to confirm your appointment for tomorrow at 2 PM.   Innocent: Oh, hi Karen. I think there might be some mistake. I don't have an appointment with Dr. Smith tomorrow.   Suspect: Okay, let me check on that. Can you please confirm your nam

PAIR 2
SW: Mtuhumiwa: Hi, huyu ni John? Innocent: Ndio mimi. Huyu ni nani? Mshukiwa: Huyu ni Karen kutoka ofisi ya Dk. Smith. Ninakupigia ili kuthibitisha miadi yako ya kesho saa 2 Usiku. Innocent: Ah, ndio. Nadhani nimepanga hiyo. Je, unaweza kuniambia zaidi kuhusu miadi hiyo? Ya nini tena? Mshukiwa: Ni ufuat
EN: Suspect: Hi, is this John? Innocent: Yeah,

In [ ]:
# ============================================================
# VishGuard — Build Clean External Conversation Dataset
# ============================================================

import pandas as pd

# First 640 = Swahili conversations
# Next 640 = English versions of the same conversations

normal_sw = normal_lines[:640]
normal_en = normal_lines[640:1280]

scam_sw = scam_lines[:640]
scam_en = scam_lines[640:1280]

# Build one row per unique conversation scenario
external_rows = []

for i in range(640):
    external_rows.append({
        "scenario_id": f"normal_{i+1:03d}",
        "type": "legitimate",
        "swahili_transcript": normal_sw[i],
        "english_transcript": normal_en[i],
        "source": "Kaggle_Voice_Phishing_Dataset_English_Swahili"
    })

for i in range(640):
    external_rows.append({
        "scenario_id": f"scam_{i+1:03d}",
        "type": "fraudulent",
        "swahili_transcript": scam_sw[i],
        "english_transcript": scam_en[i],
        "source": "Kaggle_Voice_Phishing_Dataset_English_Swahili"
    })

external_df = pd.DataFrame(external_rows)

print("=" * 80)
print("CLEAN EXTERNAL DATASET")
print("=" * 80)

print("Shape:", external_df.shape)

print("\nClass distribution:")
print(external_df["type"].value_counts())

print("\nColumns:")
print(external_df.columns.tolist())

print("\nMissing values:")
print(external_df.isna().sum())

print("\nDuplicate English transcripts:",
      external_df["english_transcript"].duplicated().sum())

print("\nDuplicate scenario IDs:",
      external_df["scenario_id"].duplicated().sum())

print("\nFirst legitimate sample:")
print(external_df.iloc[0]["english_transcript"][:1000])

print("\nFirst fraudulent sample:")
print(external_df[external_df["type"] == "fraudulent"].iloc[0]["english_transcript"][:1000])

CLEAN EXTERNAL DATASET
Shape: (1280, 5)

Class distribution:
type
legitimate    640
fraudulent    640
Name: count, dtype: int64

Columns:
['scenario_id', 'type', 'swahili_transcript', 'english_transcript', 'source']

Missing values:
scenario_id           0
type                  0
swahili_transcript    0
english_transcript    0
source                0
dtype: int64

Duplicate English transcripts: 0

Duplicate scenario IDs: 0

First legitimate sample:
Suspect: Hi, this is Karen from Dr. Smith's office. I'm calling to confirm your appointment for tomorrow at 2 PM.   Innocent: Oh, hi Karen. I think there might be some mistake. I don't have an appointment with Dr. Smith tomorrow.   Suspect: Okay, let me check on that. Can you please confirm your name and date of birth for me?   Innocent: Wait, how did you get my number? I don't recall making an appointment or giving out my contact info.   Suspect: I apologize for the confusion. We must have gotten your number from your insurance provider.   

In [ ]:
# ============================================================
# VishGuard — Create Project Folder
# ============================================================

import os

vishguard_dir = "/content/drive/MyDrive/VishGuard"

os.makedirs(vishguard_dir, exist_ok=True)

print("Folder ready:")
print(vishguard_dir)
print("Exists:", os.path.exists(vishguard_dir))

Folder ready:
/content/drive/MyDrive/VishGuard
Exists: True


In [ ]:
# ============================================================
# VishGuard — Load Swahili → English Translation Model
# ============================================================

from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer
import torch

translation_model_name = "facebook/m2m100_418M"

translation_tokenizer = M2M100Tokenizer.from_pretrained(
    translation_model_name
)

translation_model = M2M100ForConditionalGeneration.from_pretrained(
    translation_model_name
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
translation_model = translation_model.to(device)
translation_model.eval()

print("Model:", translation_model_name)
print("Device:", device)

tokenizer_config.json:   0%|          | 0.00/298 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/3.71M [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 2.42MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/1.14k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.94GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/233 [00:00<?, ?B/s]

Model: facebook/m2m100_418M
Device: cpu


In [ ]:
# ============================================================
# VishGuard — Test Swahili → English Translation
# ============================================================

def translate_swahili(text):
    translation_tokenizer.src_lang = "sw"

    encoded = translation_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(device)

    with torch.no_grad():
        generated_tokens = translation_model.generate(
            **encoded,
            forced_bos_token_id=translation_tokenizer.get_lang_id("en"),
            max_length=1024,
            num_beams=4
        )

    return translation_tokenizer.batch_decode(
        generated_tokens,
        skip_special_tokens=True
    )[0]


# Test 3 legitimate + 3 fraudulent conversations
test_indices = [0, 1, 9]

for i in test_indices:
    print("=" * 100)
    print(f"LEGITIMATE — Scenario {i+1}")
    print("-" * 100)
    print("SW:")
    print(external_df.iloc[i]["swahili_transcript"][:1000])
    print("\nTRANSLATED EN:")
    print(translate_swahili(external_df.iloc[i]["swahili_transcript"])[:1000])


for i in test_indices:
    row = external_df[external_df["type"] == "fraudulent"].iloc[i]

    print("\n" + "=" * 100)
    print(f"FRAUDULENT — Scenario {i+1}")
    print("-" * 100)
    print("SW:")
    print(row["swahili_transcript"][:1000])
    print("\nTRANSLATED EN:")
    print(translate_swahili(row["swahili_transcript"])[:1000])

LEGITIMATE — Scenario 1
----------------------------------------------------------------------------------------------------
SW:
Mshukiwa: Habari, huyu ni Karen kutoka ofisi ya Dk. Smith. Ninakupigia ili kuthibitisha miadi yako ya kesho saa 2 Usiku. Innocent: Oh, habari Karen. Nadhani kunaweza kuwa na makosa. Sina miadi na Dk. Smith kesho. Mtuhumiwa: Sawa, ngoja niangalie hilo. Tafadhali unaweza kunithibitishia jina lako na tarehe ya kuzaliwa? Innocent: Subiri, umepataje namba yangu? Sikumbuki kuweka miadi au kutoa maelezo yangu ya mawasiliano. Mtuhumiwa: Naomba radhi kwa kuchanganyikiwa. Lazima tumepata nambari yako kutoka kwa mtoa huduma wako wa bima. Innocent: Sinunui. Wewe si mzuri sana katika hili, sivyo? Mtuhumiwa: Um, sijui unamaanisha nini. Ninajaribu tu kufanya kazi yangu. Innocent: Ihifadhi. Wewe si tapeli, sivyo? Wewe ni kweli kutoka ofisi ya Dk. Smith. Mtuhumiwa: Ah, ndio. mimi niko. Naomba radhi kwa kuchanganyikiwa hapo awali. Tulikuwa na mawasiliano mabaya. Innocent: Ha! 

In [ ]:
import requests

url = "https://zenodo.org/records/20429763/files/vishguard_dataset.zip?download=1"

r = requests.head(url, allow_redirects=True)

print("Status:", r.status_code)
print("Final URL:", r.url)
print("Size:", int(r.headers.get("Content-Length", 0)) / (1024**3), "GB")

Status: 200
Final URL: https://zenodo.org/records/20429763/files/vishguard_dataset.zip?download=1
Size: 6.01077048573643 GB


In [ ]:
!df -h /

Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   48G   66G  42% /


In [ ]:
import requests
from tqdm.auto import tqdm

url = "https://zenodo.org/records/20429763/files/vishguard_dataset.zip?download=1"
output = "/content/vishguard_dataset.zip"

response = requests.get(url, stream=True)
response.raise_for_status()

total = int(response.headers.get("content-length", 0))

with open(output, "wb") as f, tqdm(
    total=total,
    unit="B",
    unit_scale=True,
    unit_divisor=1024,
    desc="Downloading"
) as pbar:
    for chunk in response.iter_content(chunk_size=1024 * 1024):
        if chunk:
            f.write(chunk)
            pbar.update(len(chunk))

print("\nDownload complete!")

Downloading:   0%|          | 0.00/6.01G [00:00<?, ?B/s]


Download complete!


In [ ]:
!unzip -l /content/vishguard_dataset.zip | head -100

Archive:  /content/vishguard_dataset.zip
  Length      Date    Time    Name
---------  ---------- -----   ----
        0  2026-05-27 20:52   vishguard_dataset/
  2691761  2025-10-04 12:36   vishguard_dataset/annotations_with_audio.json
        0  2026-02-18 10:03   vishguard_dataset/audio/
  5312504  2025-10-04 11:27   vishguard_dataset/audio/lot10_ar_fraud_1000.wav
  6054956  2025-10-04 11:24   vishguard_dataset/audio/lot10_ar_fraud_906.wav
  2458266  2025-10-04 11:24   vishguard_dataset/audio/lot10_ar_fraud_909.wav
  2919020  2025-10-04 11:24   vishguard_dataset/audio/lot10_ar_fraud_916.wav
  6070892  2025-10-04 11:24   vishguard_dataset/audio/lot10_ar_fraud_923.wav
  5470208  2025-10-04 11:25   vishguard_dataset/audio/lot10_ar_fraud_929.wav
  2720300  2025-10-04 11:25   vishguard_dataset/audio/lot10_ar_fraud_936.wav
  4832872  2025-10-04 11:25   vishguard_dataset/audio/lot10_ar_fraud_943.wav
  5398764  2025-10-04 11:25   vishguard_dataset/audio/lot10_ar_fraud_949.wav
  2469932  2025

In [ ]:
!unzip -p /content/vishguard_dataset.zip \
"vishguard_dataset/annotations_with_audio.json" \
> /content/annotations_with_audio.json

In [ ]:
import json

path = "/content/annotations_with_audio.json"

with open(path, "r", encoding="utf-8") as f:
    data = json.load(f)

print("Type:", type(data))

if isinstance(data, list):
    print("Number of records:", len(data))
    print("\nFirst record:")
    print(json.dumps(data[0], indent=2, ensure_ascii=False)[:5000])
elif isinstance(data, dict):
    print("Keys:", list(data.keys()))
    print("\nPreview:")
    print(json.dumps(data, indent=2, ensure_ascii=False)[:5000])

Type: <class 'list'>
Number of records: 3059

First record:
{
  "id": "lot1_en_fraud_001",
  "type": "fraudulent",
  "language": "en",
  "script": "Attention, your bank account has been locked due to suspected money laundering activity. An official government agent is on the line. Failure to cooperate immediately will result in a warrant for your arrest being issued within the hour. You must urgently transfer your funds to our secure verification wallet. Time is running out to avoid legal prosecution.",
  "duration": 21.01,
  "emotion": [
    "fear",
    "authority"
  ],
  "tone": [
    "threat",
    "authoritative"
  ],
  "persuasion_patterns": [
    "threat",
    "urgency",
    "authority"
  ],
  "behavior": [
    "irregular_speech",
    "interruptions"
  ],
  "metadata": {
    "words_count": 115,
    "audio_path": "audio/lot1/en_fraud_001.wav"
  },
  "audio_path": "audio\\lot1_en_fraud_001.wav"
}


In [ ]:
import pandas as pd

df = pd.DataFrame(data)

print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nClass distribution:")
print(df["type"].value_counts())

print("\nLanguage distribution:")
print(df["language"].value_counts())

print("\nClass × language:")
print(pd.crosstab(df["language"], df["type"]))

print("\nTotal duration:")
print(f"{df['duration'].sum()/3600:.2f} hours")

print("\nAverage duration:")
print(f"{df['duration'].mean():.2f} seconds")

Shape: (3059, 11)

Columns:
['id', 'type', 'language', 'script', 'duration', 'emotion', 'tone', 'persuasion_patterns', 'behavior', 'metadata', 'audio_path']

Class distribution:
type
fraudulent    1530
legitimate    1529
Name: count, dtype: int64

Language distribution:
language
en    1065
ar     998
fr     996
Name: count, dtype: int64

Class × language:
type      fraudulent  legitimate
language                        
ar               499         499
en               529         536
fr               502         494

Total duration:
18.47 hours

Average duration:
21.74 seconds


In [ ]:
# Transcribe all VISHGUARD audio files using faster-whisper

results = []

for i, record in enumerate(data, 1):

    # Build the audio path
    relative_path = record["audio_path"].replace("\\", "/")
    audio_path = "/content/vishguard_audio/vishguard_dataset/" + relative_path

    if not os.path.exists(audio_path):
        print(f"Missing: {audio_path}")
        continue

    try:
        # Transcribe audio
        segments, info = model.transcribe(
            audio_path,
            beam_size=5,
            vad_filter=True
        )

        transcript = " ".join(
            segment.text.strip()
            for segment in segments
        ).strip()

        # Store all useful metadata
        results.append({
            "id": record.get("id"),
            "type": record.get("type"),
            "language": record.get("language"),
            "script": record.get("script"),
            "duration": record.get("duration"),
            "emotion": "; ".join(record.get("emotion", [])),
            "tone": "; ".join(record.get("tone", [])),
            "persuasion_patterns": "; ".join(
                record.get("persuasion_patterns", [])
            ),
            "behavior": "; ".join(
                record.get("behavior", [])
            ),
            "audio_path": audio_path,
            "transcript": transcript,
            "whisper_language": info.language,
            "whisper_language_probability": info.language_probability
        })

    except Exception as e:
        print(f"Error processing {record.get('id')}: {e}")

    # Progress
    if i % 25 == 0:
        print(f"Processed {i}/{len(data)}")

# Convert to DataFrame
df_vishguard = pd.DataFrame(results)

print("\nTranscription completed.")
print("Records:", len(df_vishguard))
print("Columns:", list(df_vishguard.columns))

NameError: name 'data' is not defined

In [ ]:
!unzip -q /content/vishguard_dataset.zip -d /content/

In [ ]:
import os
import glob

audio_dir = "/content/vishguard_dataset/audio"

wav_files = glob.glob(os.path.join(audio_dir, "**", "*.wav"), recursive=True)

print("Number of WAV files:", len(wav_files))
print("First 10 files:")
for f in wav_files[:10]:
    print(f)

Number of WAV files: 3059
First 10 files:
/content/vishguard_dataset/audio/lot10_fr_fraud_945.wav
/content/vishguard_dataset/audio/lot24_fr_legit_2337.wav
/content/vishguard_dataset/audio/lot15_ar_legit_1495.wav
/content/vishguard_dataset/audio/lot22_fr_legit_2173.wav
/content/vishguard_dataset/audio/lot28_en_fraud_2724.wav
/content/vishguard_dataset/audio/lot27_ar_legit_2693.wav
/content/vishguard_dataset/audio/lot17_en_legit_1620.wav
/content/vishguard_dataset/audio/lot28_en_legit_2777.wav
/content/vishguard_dataset/audio/lot18_en_legit_1792.wav
/content/vishguard_dataset/audio/lot13_ar_fraud_1292.wav


In [ ]:
#Install Whisper
!pip -q install faster-whisper

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.6/39.6 MB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 49.9 MB/s eta 0:00:00


In [ ]:
!nvidia-smi

Mon Sep  7 20:00:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from faster_whisper import WhisperModel

model = WhisperModel(
    "large-v3",
    device="cuda",
    compute_type="float16"
)

print("Whisper loaded successfully.")

Whisper loaded successfully.


In [ ]:
import glob

wav_files = glob.glob("/content/vishguard_dataset/**/*.wav", recursive=True)

samples = {
    "en": next(x for x in wav_files if "_en_" in x),
    "fr": next(x for x in wav_files if "_fr_" in x),
    "ar": next(x for x in wav_files if "_ar_" in x)
}

print("Samples:")
for lang, path in samples.items():
    print(lang, "→", path)

Samples:
en → /content/vishguard_dataset/audio/lot28_en_fraud_2724.wav
fr → /content/vishguard_dataset/audio/lot10_fr_fraud_945.wav
ar → /content/vishguard_dataset/audio/lot15_ar_legit_1495.wav


In [ ]:
def transcribe(path):
    segments, info = model.transcribe(
        path,
        beam_size=5,
        vad_filter=True
    )

    text = " ".join(segment.text.strip() for segment in segments)

    return text, info

In [ ]:
for lang, path in samples.items():
    text, info = transcribe(path)

    print("\n" + "=" * 70)
    print("LANGUAGE:", lang)
    print("FILE:", path)
    print("DETECTED LANGUAGE:", info.language)
    print("ASR TRANSCRIPT:")
    print(text)


LANGUAGE: en
FILE: /content/vishguard_dataset/audio/lot28_en_fraud_2724.wav
DETECTED LANGUAGE: en
ASR TRANSCRIPT:
urgent your social security number was flagged for fraudulent activity related to a mortgage application i am the u.s social security administration officer you face immediate fraud charges to halt the legal action you must immediately purchase six dollar 850 feet immunity vouchers using steam gift cards obey my authority now to secure this one-time chance to clear your name

LANGUAGE: fr
FILE: /content/vishguard_dataset/audio/lot10_fr_fraud_945.wav
DETECTED LANGUAGE: fr
ASR TRANSCRIPT:
Je suis le chef de la sécurité sociale. Votre numéro de sécurité sociale a été signalé pour fraude et vol d'identité. Ceci est une infraction fédérale grave. Je suis le seul agent avec l'autorité de suspendre votre dossier. Vous devez transférer une caution de 500 euros à notre compte bloqué. N'en parlez à personne. Votre liberté est en jeu. Obéissez à mon autorité immédiatement.

LANGUAGE:

In [ ]:
!pip -q install jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 49.4 MB/s eta 0:00:00


In [ ]:
import json
import re
from jiwer import wer

with open("/content/annotations_with_audio.json", "r", encoding="utf-8") as f:
    records = json.load(f)

record_by_id = {r["id"]: r for r in records}

def normalize_text(text):
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text

for lang, path in samples.items():
    filename = path.split("/")[-1]
    sample_id = filename.rsplit(".", 1)[0]

    reference = record_by_id[sample_id]["script"]
    hypothesis, info = transcribe(path)

    score = wer(
        normalize_text(reference),
        normalize_text(hypothesis)
    )

    print("\n" + "=" * 70)
    print("LANGUAGE:", lang)
    print("ID:", sample_id)
    print(f"WER: {score:.4f} ({score*100:.2f}%)")
    print("\nREFERENCE:")
    print(reference)
    print("\nASR:")
    print(hypothesis)


LANGUAGE: en
ID: lot28_en_fraud_2724
WER: 0.2241 (22.41%)

REFERENCE:
Urgent! Your social security number was flagged for fraudulent activity related to a mortgage application. I am the US Social Security Administration Officer. You face immediate fraud charges. To halt the legal action, you must immediately purchase six $850 'Immunity Vouchers' using Steam gift cards. Obey my authority now to secure this one-time chance to clear your name.

ASR:
urgent your social security number was flagged for fraudulent activity related to a mortgage application i am the u.s social security administration officer you face immediate fraud charges to halt the legal action you must immediately purchase six dollar 850 feet immunity vouchers using steam gift cards obey my authority now to secure this one-time chance to clear your name

LANGUAGE: fr
ID: lot10_fr_fraud_945
WER: 0.0317 (3.17%)

REFERENCE:
Je suis le Chef de la Sécurité Sociale. Votre numéro de sécurité sociale a été signalé pour fraude et

In [ ]:
#----------------Now: transcribe the entire dataset----------------------
import pandas as pd
import os

df = pd.DataFrame(records)

df["asr_transcript"] = ""
df["asr_detected_language"] = ""
df["asr_status"] = "pending"

output_file = "/content/vishguard_asr_transcriptions.csv"

print("Records:", len(df))
print("Output:", output_file)

Records: 3059
Output: /content/vishguard_asr_transcriptions.csv


In [ ]:
import os
import time
import pandas as pd
from tqdm.auto import tqdm

output_file = "/content/vishguard_asr_transcriptions.csv"

# Load previous progress if it exists
if os.path.exists(output_file):
    df = pd.read_csv(output_file)
    print("Loaded existing progress.")
else:
    df = pd.DataFrame(records)
    df["asr_transcript"] = ""
    df["asr_detected_language"] = ""
    df["asr_status"] = "pending"

print("Total records:", len(df))
print("Already completed:", (df["asr_status"] == "completed").sum())
print("Remaining:", (df["asr_status"] != "completed").sum())

Total records: 3059
Already completed: 0
Remaining: 3059


In [ ]:
start_time = time.time()

for i in tqdm(range(len(df)), desc="Transcribing"):

    # Skip files already processed
    if df.at[i, "asr_status"] == "completed":
        continue

    audio_path = df.at[i, "audio_path"]

    # Convert Windows-style path if necessary
    audio_path = audio_path.replace("\\", "/")

    # The JSON's top-level path is like:
    # audio/lot1_en_fraud_001.wav
    full_path = os.path.join(
        "/content/vishguard_dataset",
        audio_path
    )

    # Safety check
    if not os.path.exists(full_path):
        df.at[i, "asr_status"] = "file_not_found"
        continue

    try:
        segments, info = model.transcribe(
            full_path,
            beam_size=5,
            vad_filter=True
        )

        text = " ".join(
            segment.text.strip()
            for segment in segments
        )

        df.at[i, "asr_transcript"] = text
        df.at[i, "asr_detected_language"] = info.language
        df.at[i, "asr_status"] = "completed"

    except Exception as e:
        df.at[i, "asr_status"] = f"error: {str(e)[:150]}"

    # Save every 25 files
    if (i + 1) % 25 == 0:
        df.to_csv(output_file, index=False)

# Final save
df.to_csv(output_file, index=False)

elapsed = time.time() - start_time

print("\nFinished!")
print(f"Time: {elapsed/60:.1f} minutes")
print("Completed:", (df["asr_status"] == "completed").sum())
print("Errors:", df["asr_status"].astype(str).str.startswith("error").sum())
print("Missing:", (df["asr_status"] == "file_not_found").sum())
print("Saved to:", output_file)

Transcribing:   0%|          | 0/3059 [00:00<?, ?it/s]


Finished!
Time: 138.4 minutes
Completed: 3059
Errors: 0
Missing: 0
Saved to: /content/vishguard_asr_transcriptions.csv


In [ ]:
import pandas as pd

asr_df = pd.read_csv("/content/vishguard_asr_transcriptions.csv")

print("Shape:", asr_df.shape)
print("\nColumns:")
print(asr_df.columns.tolist())

print("\nMissing ASR transcripts:",
      asr_df["asr_transcript"].isna().sum())

print("\nEmpty ASR transcripts:",
      (asr_df["asr_transcript"].fillna("").str.strip() == "").sum())

print("\nDetected languages:")
print(asr_df["asr_detected_language"].value_counts())

print("\nStatus:")
print(asr_df["asr_status"].value_counts())

Shape: (3059, 14)

Columns:
['id', 'type', 'language', 'script', 'duration', 'emotion', 'tone', 'persuasion_patterns', 'behavior', 'metadata', 'audio_path', 'asr_transcript', 'asr_detected_language', 'asr_status']

Missing ASR transcripts: 0

Empty ASR transcripts: 0

Detected languages:
asr_detected_language
en    1065
ar     998
fr     996
Name: count, dtype: int64

Status:
asr_status
completed    3059
Name: count, dtype: int64


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import shutil

drive_folder = "/content/drive/MyDrive/VishGuard"

os.makedirs(drive_folder, exist_ok=True)

print("Folder ready:", drive_folder)

Folder ready: /content/drive/MyDrive/VishGuard


In [ ]:
source = "/content/vishguard_asr_transcriptions.csv"
destination = "/content/drive/MyDrive/VishGuard/vishguard_asr_transcriptions.csv"

shutil.copy2(source, destination)

print("Backup saved successfully!")
print(destination)

Backup saved successfully!
/content/drive/MyDrive/VishGuard/vishguard_asr_transcriptions.csv
